In [1]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from peft import PeftModel
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.stats import binomtest

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [2]:
# Path to your saved model
MODEL_DIR = "../results/models/lora_b1_final"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# Load base model + LoRA adapter
base_model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
)
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()
model.to(device)

print(f"✅ Model loaded from: {MODEL_DIR}")
print(f"Model type: {type(model).__name__}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded from: ../results/models/lora_b1_final
Model type: PeftModelForSequenceClassification


In [3]:
test_df = pd.read_csv("../data/processed/test_hard2.csv")

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print(f"Test size: {len(test_df)}")
print(f"Label distribution:\n{test_df['label'].value_counts()}")
print(f"\nSample SMS:")
for t in test_df['text_clean'].head(3):
    print(f"  - {t[:100]}")

Test size: 2107
Label distribution:
label
smish     1267
normal     498
promo      342
Name: count, dtype: int64

Sample SMS:
  - আপনি ৩ vori gold জিতেছেন! Claim করুন: [PHONE]
  - জয়! আপনি Google লটারিতে $৫০০ জিতেছেন। claim করতে: winlottery.tk/claim পূরণ করুন।
  - জনতা ব্যাংক অ্যাকাউন্ট ব্লক হয়েছে। পুনরায় চালু করতে কল করুন: [PHONE]


In [4]:
class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len,
            return_tensors=None,
        )
        return enc

texts = test_df['text_clean'].astype(str).tolist()
y_true = np.array([label2id[l] for l in test_df['label']])

test_ds = SMSDataset(texts, tokenizer)
test_loader = DataLoader(
    test_ds, batch_size=32,
    collate_fn=DataCollatorWithPadding(tokenizer),
)

# Predictions
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

y_pred = np.array(all_preds)
y_probs = np.array(all_probs)

print(f"✅ Predictions generated")
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 : {f1_score(y_true, y_pred, average='macro'):.4f}")

✅ Predictions generated
Accuracy : 0.7451
Macro F1 : 0.7463


In [5]:
smish_idx = label2id['smish']
smish_mask = y_true == smish_idx
misclassified = smish_mask & (y_pred != smish_idx)

print("=" * 60)
print("ERROR ANALYSIS — SMISH DETECTION")
print("=" * 60)
print(f"Total smish samples   : {smish_mask.sum()}")
print(f"Correctly classified  : {(y_pred[smish_mask] == smish_idx).sum()}")
print(f"Misclassified as other: {misclassified.sum()}")
print(f"Miss rate             : {misclassified.sum()/smish_mask.sum()*100:.1f}%")

# What are they classified as?
error_df = test_df[misclassified].copy()
error_df['predicted_as'] = [labels[p] for p in y_pred[misclassified]]

print(f"\nMisclassified smish predicted as:")
print(error_df['predicted_as'].value_counts())

ERROR ANALYSIS — SMISH DETECTION
Total smish samples   : 1267
Correctly classified  : 746
Misclassified as other: 521
Miss rate             : 41.1%

Misclassified smish predicted as:
predicted_as
normal    307
promo     214
Name: count, dtype: int64


In [6]:
# Categorize errors by pattern
error_df = test_df[misclassified].copy()
error_df['predicted_as'] = [labels[p] for p in y_pred[misclassified]]

def categorize(text):
    text = str(text).lower()
    cats = []
    if '[url]' not in text:
        cats.append('no_url')
    if '[phone]' not in text:
        cats.append('no_phone')
    if len(text) < 50:
        cats.append('short_text')
    if any(c.isdigit() for c in text):
        cats.append('has_numbers')
    if not any('\u0980' <= c <= '\u09FF' for c in text):
        cats.append('no_bangla')
    return cats

error_df['categories'] = error_df['text_clean'].apply(categorize)

# Count each category
cat_counts = Counter()
for cats in error_df['categories']:
    for c in cats:
        cat_counts[c] += 1

print("=" * 60)
print("ERROR CATEGORIES")
print("=" * 60)
total_errors = len(error_df)
for cat, count in cat_counts.most_common():
    print(f"  {cat:15s}: {count:4d} ({count/total_errors*100:5.1f}%)")

ERROR CATEGORIES
  no_url         :  521 (100.0%)
  no_bangla      :  309 ( 59.3%)
  has_numbers    :  262 ( 50.3%)
  no_phone       :  144 ( 27.6%)
  short_text     :   65 ( 12.5%)


In [7]:
print("=" * 60)
print("SAMPLE MISCLASSIFIED SMS (10 examples)")
print("=" * 60)

for i, (_, row) in enumerate(error_df.sample(10, random_state=SEED).iterrows(), 1):
    print(f"\n{i}. True: smish | Predicted: {row['predicted_as']}")
    print(f"   Text: {row['text_clean'][:200]}")

SAMPLE MISCLASSIFIED SMS (10 examples)

1. True: smish | Predicted: promo
   Text: প্রিয় আবেদনকারী, মঙ্গল 04/05 তারিখে সকাল 9 টায় @1ম তলায় কেন্দ্র শত্রু নেতৃত্বের সাথে একটি ইন্টার-ভিউয়ের জন্য আপনাকে বাছাই করা হয়েছে। অনুসন্ধানের জন্য অনুগ্রহ করে যোগাযোগ করুন: [PHONE]

2. True: smish | Predicted: normal
   Text: সোনালী ব্যাংক থেকে আপনার জন্য জরুরি বার্তা। কল করুন: [PHONE]

3. True: smish | Predicted: promo
   Text: আপনি আমাদের lucky customer! আপনার free 1 বছরের internet subscription claim করুন। Call করুন 019XX।

4. True: smish | Predicted: normal
   Text: আপনার account-এ problem হয়েছে। দ্রুত এই number-এ যোগাযোগ করুন [PHONE] ৃ।

5. True: smish | Predicted: promo
   Text: অনলাইনে সার্চ দিন প্রিয় কেসিনো এবং জিতে নিন হাজার হাজার বোনাস মাত্র কয়েক ঘন্টায়।

6. True: smish | Predicted: promo
   Text: Bekar jubok, juboti, grihini, retired bektigder jonno gore bose online e sholpo somoy e prochur taka uparjon korar shuborgo sujog

7. True: smish | Predicted: promo
   Text: বেকার যুবক, যুবতী,

In [8]:
# Save
os.makedirs("../results/analysis", exist_ok=True)
error_df.to_csv("../results/analysis/misclassified_smish.csv", index=False)

error_summary = {
    "total_smish": int(smish_mask.sum()),
    "correctly_classified": int((y_pred[smish_mask] == smish_idx).sum()),
    "misclassified": int(misclassified.sum()),
    "miss_rate_pct": float(misclassified.sum()/smish_mask.sum()*100),
    "error_categories": {k: int(v) for k, v in cat_counts.most_common()},
    "predicted_distribution": error_df['predicted_as'].value_counts().to_dict(),
}

with open("../results/analysis/error_summary.json", "w") as f:
    json.dump(error_summary, f, indent=2)

print("✅ Saved: error_summary.json + misclassified_smish.csv")

✅ Saved: error_summary.json + misclassified_smish.csv


In [9]:
# ============================================================
# PART 2: STATISTICAL SIGNIFICANCE TEST
# ============================================================
print("=" * 60)
print("PART 2: McNEMAR'S TEST")
print("=" * 60)

# ---------- Step 2.1: Train Baseline ----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.stats import binomtest

# Load training data
train_df = pd.read_csv("../data/processed/train_hard2.csv")

print(f"Train size: {train_df.shape}")
print(f"Test size : {test_df.shape}")

# Baseline pipeline
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=SEED,
    )),
])

baseline.fit(train_df['text_clean'], train_df['label'])
baseline_pred_labels = baseline.predict(test_df['text_clean'])
baseline_pred = np.array([label2id[p] for p in baseline_pred_labels])

from sklearn.metrics import f1_score
baseline_f1 = f1_score(y_true, baseline_pred, average='macro')
lora_f1 = f1_score(y_true, y_pred, average='macro')

print(f"\nBaseline Test F1: {baseline_f1:.4f}")
print(f"LoRA Test F1    : {lora_f1:.4f}")
print(f"Improvement     : {lora_f1 - baseline_f1:+.4f}")

PART 2: McNEMAR'S TEST
Train size: (4898, 6)
Test size : (2107, 6)

Baseline Test F1: 0.5963
LoRA Test F1    : 0.7463
Improvement     : +0.1500


In [10]:
# ---------- Step 2.2: McNemar's Test ----------
print("\n" + "=" * 60)
print("McNEMAR'S TEST — Baseline vs LoRA")
print("=" * 60)

# Which samples each model got right?
correct_baseline = (baseline_pred == y_true)
correct_lora = (y_pred == y_true)

# 2x2 Contingency table
a = (correct_baseline & correct_lora).sum()    # both correct
b = (correct_baseline & ~correct_lora).sum()   # only baseline correct
c = (~correct_baseline & correct_lora).sum()   # only LoRA correct
d = (~correct_baseline & ~correct_lora).sum()  # both wrong

print(f"\n{'':20s} | {'LoRA Correct':>15s} | {'LoRA Wrong':>12s}")
print(f"{'-'*52}")
print(f"{'Baseline Correct':20s} | {a:>15d} | {b:>12d}")
print(f"{'Baseline Wrong':20s} | {c:>15d} | {d:>12d}")

print(f"\nLoRA advantage: {c} - {b} = {c-b} samples")

# McNemar's test (exact binomial)
if b + c > 0:
    result = binomtest(c, b + c)
    
    print(f"\nMcNemar's Test:")
    print(f"  Test statistic (c): {c}")
    print(f"  b + c             : {b + c}")
    print(f"  p-value           : {result.pvalue:.2e}")
    
    if result.pvalue < 0.001:
        print(f"\n  ✅ HIGHLY SIGNIFICANT (p < 0.001)")
    elif result.pvalue < 0.01:
        print(f"\n  ✅ VERY SIGNIFICANT (p < 0.01)")
    elif result.pvalue < 0.05:
        print(f"\n  ✅ SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n  ⚠️ Not significant (p >= 0.05)")


McNEMAR'S TEST — Baseline vs LoRA

                     |    LoRA Correct |   LoRA Wrong
----------------------------------------------------
Baseline Correct     |            1106 |           58
Baseline Wrong       |             464 |          479

LoRA advantage: 464 - 58 = 406 samples

McNemar's Test:
  Test statistic (c): 464
  b + c             : 522
  p-value           : 1.11e-79

  ✅ HIGHLY SIGNIFICANT (p < 0.001)


In [11]:
# ---------- Step 2.3: Save Results ----------
import json
import os

os.makedirs("../results/analysis", exist_ok=True)

stats_results = {
    "baseline_f1": float(baseline_f1),
    "lora_f1": float(lora_f1),
    "improvement": float(lora_f1 - baseline_f1),
    "contingency": {
        "both_correct": int(a),
        "only_baseline_correct": int(b),
        "only_lora_correct": int(c),
        "both_wrong": int(d),
    },
    "mcnemar": {
        "test_statistic": int(c),
        "p_value": float(result.pvalue),
        "significant_at_0.05": bool(result.pvalue < 0.05),
        "significant_at_0.001": bool(result.pvalue < 0.001),
    },
}

with open("../results/analysis/statistical_test.json", "w") as f:
    json.dump(stats_results, f, indent=2)

print("✅ Saved: ../results/analysis/statistical_test.json")
print("\n" + json.dumps(stats_results, indent=2))

✅ Saved: ../results/analysis/statistical_test.json

{
  "baseline_f1": 0.5963430423982029,
  "lora_f1": 0.746337323978222,
  "improvement": 0.14999428158001904,
  "contingency": {
    "both_correct": 1106,
    "only_baseline_correct": 58,
    "only_lora_correct": 464,
    "both_wrong": 479
  },
  "mcnemar": {
    "test_statistic": 464,
    "p_value": 1.1118264033422418e-79,
    "significant_at_0.05": true,
    "significant_at_0.001": true
  }
}


In [ ]:
# ---------- Step 2.4: Paper Summary ----------
print("=" * 70)
print("PAPER SUMMARY — Statistical Significance")
print("=" * 70)
print(f"{'Model':30s} | {'Macro F1':>10s}")
print("-" * 50)
print(f"{'Baseline (TF-IDF+LR)':30s} | {baseline_f1:>10.4f}")
print(f"{'LoRA + XLM-R (Ours)':30s} | {lora_f1:>10.4f}")
print(f"{'Improvement':30s} | {lora_f1 - baseline_f1:>+10.4f}")
print("=" * 50)
print(f"\nMcNemar's Test p-value: {result.pvalue:.2e}")
print(f"Significant at α=0.001: {'✅ YES' if result.pvalue < 0.001 else '❌ NO'}")

print("\n" + "=" * 70)
print("📝 Paper-এর জন্য বাক্য:")
print("=" * 70)
print(f"""
"The improvement of LoRA over the TF-IDF baseline is statistically 
significant (McNemar's test, p < 0.001), with LoRA correctly 
classifying {c} samples that the baseline missed, compared to 
only {b} samples where the reverse occurred."
""")

PAPER SUMMARY — Statistical Significance
Model                          |   Macro F1
--------------------------------------------------
Baseline (TF-IDF+LR)           |     0.5963
LoRA + XLM-R (Ours)            |     0.7463
Improvement                    |    +0.1500

McNemar's Test p-value: 1.11e-79
Significant at α=0.001: ✅ YES

📝 Paper-এর জন্য বাক্য:

"The improvement of LoRA over the TF-IDF baseline is statistically 
significant (McNemar's test, p < 0.001), with LoRA correctly 
classifying 464 samples that the baseline missed, compared to 
only 58 samples where the reverse occurred."



: 